In [1]:
import folium
from folium.plugins import TimeSliderChoropleth, TimestampedGeoJson
import geopandas as gpd
import pandas as pd
import branca.colormap as cm
import warnings
import numpy as np
from shapely import geometry
import matplotlib as mpl
import matplotlib.cm as cm1
from matplotlib.colors import to_hex

warnings.filterwarnings("ignore")

/home/wojtek/.local/lib/python3.8/site-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (1.25.10) or chardet (4.0.0) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


# Zadanie 1

In [3]:
gdf = gpd.read_file("folium/country.geojson")
covid = pd.read_csv('folium/owid-covid-data.csv')
m = folium.Map(
    location=[53.4327, 14.5483],
    zoom_start=2,
    tiles='OpenStreetMap',
    crs='EPSG3857')
folium.GeoJson(
        gdf,
        name="Państwa",
        smooth_factor=2
    ).add_to(m)
m.save(outfile='zad1.html')

# Zadanie 2

In [4]:
gdf = gpd.read_file("folium/country.geojson")

covid = pd.read_csv('folium/owid-covid-data.csv')
covid = covid[~covid.iso_code.str.startswith('OWID')]
new_covid = gpd.GeoDataFrame(columns=['iso_code', 'date', 'total_cases_per_million'])
i = 0
for country in covid["iso_code"].unique():
    most_recent_date = max(covid[covid['iso_code'] == country]['date'])
    new_covid.loc[i] = [country,
                        most_recent_date,
                        covid[(covid['iso_code'] == country) & (covid['date'] == most_recent_date)]['total_cases_per_million'].values[0]]
    i += 1

gdf_merged = gdf.merge(new_covid, left_on='GMI_CNTRY', right_on='iso_code')

m = folium.Map(
    location=[53.4327, 14.5483],
    zoom_start=2,
    tiles='OpenStreetMap',
    crs='EPSG3857')

bins = list(new_covid['total_cases_per_million'].quantile([0, 0.4, 0.6, 0.8, 0.9, 1]))
print(bins)
bins1 = list(new_covid['total_cases_per_million'].quantile([x/10 for x in range(0, 11, 2)]))
print(bins1)

choropleth = folium.Choropleth(
    geo_data=gdf_merged,
    name='Liczba zachorowań',
    data=new_covid,
    columns=['iso_code', 'total_cases_per_million'],
    key_on='feature.properties.GMI_CNTRY',
    fill_color='OrRd',
    fill_opacity=0.7,
    line_opacity=0.1,
    legend_name='Liczba zachorowań',
    bins=bins,
    reset=True,
    highlight=True,
).add_to(m)
test = folium.plugins.TimestampedGeoJson(None)
test.add_child(folium.GeoJsonTooltip(
    fields=['total_cases_per_million']
))
folium.GeoJson(
    gdf_merged,
    name="Państwa",
    smooth_factor=2,
    style_function=lambda x: {'color': 'black', 'fillColor': 'transparent', 'weight': 2},
    tooltip=folium.GeoJsonTooltip(fields=['total_cases_per_million'], labels=False, sticky=False),
    highlight_function=lambda x: {'weight': 3, 'fillColor': 'gray'}
).add_to(m)

folium.LayerControl().add_to(m)
m.save(outfile='zad2.html')

[6.597, 4724.8136, 20649.383199999997, 49112.753600000004, 66366.98600000005, 145318.061]
[6.597, 951.2583999999999, 4724.8136, 20649.383199999997, 49112.753600000004, 145318.061]
